# Digital workflow on a public PMD infrastructure

## Overview

Three complementary RDF datasets describe tensile tests on S355 steel. We start with existing semantic annotations and use them to inspect contributions, assemble a scientific artifact, and prepare a publication.

| Walkthrough question | What we will do | Time |
| --- | --- | --- |
| 1. What is available? | Discover contributions and add or reuse our dataset | 10 min |
| 2. What does it mean? | Query material, orientation, files, quantities and units | 15 min |
| 3. What belongs together? | Combine the three selected graphs | 5 min |
| 4. Which metadata can we reuse? | Extract facts and review publication choices | 5 min |
| 5. What can others access? | Publish or inspect a record, then verify its resources | 10 min |

The workflow prepares data for reuse. Creating the original annotations, calculating mechanical properties and generating figures are outside this walkthrough.

**Saved-output notice:** outputs from the preceding execution have been preserved. They do not yet reflect the revised code and display formats. Run a fresh, reviewed execution before presenting, then remove this notice.

## Infrastructure and responsibilities

- **Ontodocker** hosts the three research graphs and exposes SPARQL endpoints.
- **MaterialDigital Dataportal** holds the publication record, the uploaded Turtle artifact, links to external CSV files, and generated metadata resources.
- **praeco** provides the Python clients used to access both services.

Shared concepts make the contributions interpretable together. The researcher still decides what to publish, who to credit, which organization owns the record, and which reuse terms apply.

## Getting started

### Environment

Start Jupyter in `pmd_public-infrastructure-demonstrator/notebooks` and select the `pmd-demonstrator` kernel. The setup below checks this working directory and defines all data paths from it.

Set `ONTODOCKER_ADDRESS` and `DATAPORTAL_TOKEN` before starting Jupyter. Set `ONTODOCKER_TOKEN` if required by the service. Credentials can be loaded from a local environment file; never print tokens or save them in notebook outputs.

The default settings reuse existing remote datasets. Local assembly writes a Turtle file; downloading and querying are network reads. Changing a mode to `create` enables the labelled remote creation/upload cells. Cleanup is kept outside the walkthrough.

#### Verify the environment

In [1]:
import os

def require_env(name: str) -> str:
    value = os.getenv(name)
    if value is None or not value.strip():
        raise RuntimeError(f"Required environment variable {name} is not set.")
    return value.strip()

ontodocker_address = require_env("ONTODOCKER_ADDRESS")
ontodocker_token = os.getenv("ONTODOCKER_TOKEN") or None

dataportal_token = require_env("DATAPORTAL_TOKEN")

In [2]:
ontodocker_address  # Display the service address, never the tokens.

'ontodocker.material-digital.de'

### Walkthrough settings

Keep both modes at `reuse` to inspect the existing demonstration. For a new live publication, choose `create` and a new publication name; inspect the metadata before setting `publication_reviewed = True`. A name that already exists is never overwritten automatically.

The Ontodocker contribution can be reused independently of the publication. Use `create` there only with an unused dataset name. After changing settings, restart from this section so later cells use the same configuration.

In [3]:
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

notebook_dir = Path.cwd()
demo_dir = notebook_dir.parent
if notebook_dir.name != "notebooks" or not (demo_dir / "query_collection.py").is_file():
    raise RuntimeError("Start the kernel in pmd_public-infrastructure-demonstrator/notebooks.")

ontodocker_mode = "create"  # "reuse" or "create"
publication_mode = "reuse"  # "reuse" or "create"
ontodocker_dataset_name = "pmdco2_tto_example_diagonal"
publication_name = "pmdco2-tto-recombined"
owner_org = "mpi-susmat"  # Confirm this organization before creating a publication.
publication_reviewed = False

selected_dataset_names = (
    "pmdco2_tto_example_parallel",
    "pmdco2_tto_example_perpendicular",
    ontodocker_dataset_name,
)
if len(set(selected_dataset_names)) != 3:
    raise ValueError("Select three distinct contributions.")
if ontodocker_mode not in {"reuse", "create"} or publication_mode not in {"reuse", "create"}:
    raise ValueError("Modes must be 'reuse' or 'create'.")

data_dir = demo_dir / "datasets"
downloaded_dir = demo_dir / "downloaded"
ttl_path = data_dir / "pmdco2_tto_recombined.ttl"

## 1. What is available?

### Access Ontodocker via `praeco`

We first inspect what collaborators have made available. The example uses three datasets on one Ontodocker instance.

In [4]:
from praeco import OntodockerClient

ontodocker = OntodockerClient(address=ontodocker_address, token=ontodocker_token)
ontodocker.base_url

'https://ontodocker.material-digital.de'

#### Explore datasets and (SPARQL-)endpoints

In [5]:
available_dataset_names = ontodocker.datasets.list()
pd.DataFrame({"Dataset": available_dataset_names})

,Dataset
0,pmdco2_tto_example_parallel
1,pmdco2_tto_example_perpendicular


In [6]:
available_endpoints = ontodocker.endpoints.list()
pd.DataFrame(
    [{"Dataset": ep.dataset, "SPARQL endpoint": ep.sparql_endpoint} for ep in available_endpoints]
)

,Dataset,SPARQL endpoint
0,pmdco2_tto_example_parallel,https://ontodocker.material-digital.de/api/v1/...
1,pmdco2_tto_example_perpendicular,https://ontodocker.material-digital.de/api/v1/...


### Add or reuse our contribution

Two contributions describe parallel and perpendicular specimens; our contribution describes diagonal specimens. In `reuse` mode we require our dataset to exist. In `create` mode we create an empty dataset only if its name is unused.

**Execution:** network read in `reuse` mode; remote creation in `create` mode.

In [7]:
pending_upload_name = None
available_dataset_names = ontodocker.datasets.list()
if ontodocker_mode == "reuse":
    if ontodocker_dataset_name not in available_dataset_names:
        raise RuntimeError("The selected contribution does not exist. Review the name and mode.")
    contribution_status = "Reusing existing contribution"
else:
    if ontodocker_dataset_name in available_dataset_names:
        raise RuntimeError("Dataset already exists. Choose reuse or a new name; no data were changed.")
    ontodocker.datasets.create(ontodocker_dataset_name)
    pending_upload_name = ontodocker_dataset_name
    contribution_status = "Created empty contribution; upload follows"

pd.DataFrame([{"Dataset": ontodocker_dataset_name, "Status": contribution_status}])

,Dataset,Status
0,pmdco2_tto_example_diagonal,Created empty contribution; upload follows


The local Turtle file contains the semantic description of the diagonal contribution. Uploading it populates the newly created graph. Reusing a dataset leaves its remote content unchanged.

In [8]:
turtle_path = data_dir / "pmdco2_tto_example_diagonal.ttl"
if not turtle_path.is_file():
    raise FileNotFoundError(turtle_path)
turtle_path

PosixPath('/home/mbruns/projects/pmd-demonstrator/pmd-demonstrator/pmd_public-infrastructure-demonstrator/datasets/pmdco2_tto_example_diagonal.ttl')

In [9]:
if ontodocker_mode == "create":
    if pending_upload_name != ontodocker_dataset_name:
        raise RuntimeError("No new dataset is awaiting upload. Use reuse to inspect an existing contribution.")
    ontodocker.datasets.upload_turtlefile(ontodocker_dataset_name, turtle_path)
    pending_upload_name = None
    contribution_status = "Uploaded diagonal RDF graph"
else:
    contribution_status = "Reused existing graph; no upload"

pd.DataFrame([{"Dataset": ontodocker_dataset_name, "Status": contribution_status}])

,Dataset,Status
0,pmdco2_tto_example_diagonal,Uploaded diagonal RDF graph


## 2. What does it mean?

### Ask research questions with SPARQL

A list of files is only the starting point. We now connect measurements to their material, specimen orientation, physical quantities and units.

`query_collection.py` contains named domain queries and their result-column definitions. We inspect one SPARQL query, then concentrate on its answer. These queries depend on the shared domain model; the same question can be reused across compatible graphs.

The compact displays shorten identifiers for readability. The original query result remains in `result`, with complete URIs available for inspection.

In [10]:
from html import escape
from urllib.parse import urlsplit

pd.set_option("display.max_colwidth", 70)


def local_name(value):
    """Shorten a URI for display without changing the query result."""
    return str(value).rstrip("/").rsplit("/", 1)[-1].rsplit("#", 1)[-1]


def resource_links(rows):
    """Display short labels with clickable HTTP(S) resource links."""
    cells = []
    for label, url in rows:
        if urlsplit(url).scheme not in {"http", "https"}:
            raise ValueError("Resource links must use HTTP(S).")
        cells.append(
            f'<tr><td>{escape(label)}</td>'
            f'<td><a href="{escape(url, quote=True)}" target="_blank" rel="noopener">Open resource</a></td></tr>'
        )
    display(HTML("<table><tr><th>Resource</th><th>Link</th></tr>" + "".join(cells) + "</table>"))

In [11]:
import sys

query_collection_dir = str(demo_dir)
if query_collection_dir not in sys.path:
    sys.path.insert(0, query_collection_dir)

from query_collection import S355queries

In [12]:
query = S355queries.material_designation()

In [13]:
print(query.query)


        PREFIX pmd: <https://w3id.org/pmd/co/>
        SELECT DISTINCT (?p AS ?uri) (?matDesVal AS ?materialDesignation)
        WHERE {
            ?s a pmd:TestPiece .
            ?p pmd:input ?s .
            ?p pmd:characteristic ?matDes .
            ?matDes a pmd:materialDesignation .
            ?matDes pmd:value ?matDesVal .
        }
        ORDER BY ?p
        


In [14]:
query.query_vars

['uri', 'materialDesignation']

Run the material question against our contribution. The answer should connect each test to S355 steel.

In [15]:
result = ontodocker.sparql.query_df(ontodocker_dataset_name, query.query, query.query_vars)
result.assign(uri=result["uri"].map(local_name)).rename(
    columns={"uri": "Test", "materialDesignation": "Material"}
)

,Test,Material
0,pmdao-tto-tt-S355-10_process,S355
1,pmdao-tto-tt-S355-9_process,S355


In [16]:
query = S355queries.orientation()
result = ontodocker.sparql.query_df(ontodocker_dataset_name, query.query, query.query_vars)
result.assign(uri=result["uri"].map(local_name)).rename(
    columns={"uri": "Test", "cut_orientation": "Specimen orientation"}
)

,Test,Specimen orientation
0,pmdao-tto-tt-S355-10_process,diagonal to rolling direction
1,pmdao-tto-tt-S355-9_process,diagonal to rolling direction


In [17]:
query = S355queries.csv_url()
result = ontodocker.sparql.query_df(ontodocker_dataset_name, query.query, query.query_vars)
resource_links([
    (f"{local_name(row.uri)} — {local_name(row.url)}", str(row.url))
    for row in result.itertuples(index=False)
])

Resource,Link
pmdao-tto-tt-S355-10_process — Zd3.csv,Open resource
pmdao-tto-tt-S355-9_process — Zd2.csv,Open resource


In [18]:
query = S355queries.csv_columns()
result = ontodocker.sparql.query_df(ontodocker_dataset_name, query.query, query.query_vars)
column_summary = result[["uri", "url", "column_number", "quantity", "unit"]].copy()
for column in ("uri", "url", "quantity", "unit"):
    column_summary[column] = column_summary[column].map(local_name)
column_summary.rename(columns={
    "uri": "Test", "url": "CSV file", "column_number": "Column",
    "quantity": "Quantity", "unit": "Unit",
})

,Test,CSV file,Column,Quantity,Unit
0,pmdao-tto-tt-S355-10_process,Zd3.csv,1,Duration,SEC
1,pmdao-tto-tt-S355-10_process,Zd3.csv,2,CrossheadSeparation,MilliM
2,pmdao-tto-tt-S355-10_process,Zd3.csv,3,Force,KiloN
3,pmdao-tto-tt-S355-10_process,Zd3.csv,4,PercentageExtension,PERCENT
4,pmdao-tto-tt-S355-9_process,Zd2.csv,1,Duration,SEC
5,pmdao-tto-tt-S355-9_process,Zd2.csv,2,CrossheadSeparation,MilliM
6,pmdao-tto-tt-S355-9_process,Zd2.csv,3,Force,KiloN
7,pmdao-tto-tt-S355-9_process,Zd2.csv,4,PercentageExtension,PERCENT


**Interpretation:** the diagonal tests link to `Zd2.csv` and `Zd3.csv`. Their third column represents force in kilonewtons (`KiloN`). These descriptions make the measurement columns interpretable in their experimental context.

**Discussion:** what would be missing if a colleague received only the CSV files?

### Ask the same question of the three selected contributions

In [19]:
endpoints_by_name = {ep.dataset: ep for ep in ontodocker.endpoints.list()}
missing_datasets = set(selected_dataset_names) - endpoints_by_name.keys()
if missing_datasets:
    raise RuntimeError(f"Selected contributions are missing: {sorted(missing_datasets)}")
selected_endpoints = [endpoints_by_name[name] for name in selected_dataset_names]

In [20]:
query = S355queries.material_designation()
contribution_results = []
for endpoint in selected_endpoints:
    contribution_result = ontodocker.sparql.query_df(
        endpoint.dataset, query.query, query.query_vars
    )
    contribution_results.append(contribution_result.assign(contribution=endpoint.dataset))

all_tests = pd.concat(contribution_results, ignore_index=True)
all_tests.assign(uri=all_tests["uri"].map(local_name)).rename(columns={
    "contribution": "Contribution", "uri": "Test", "materialDesignation": "Material",
})[["Contribution", "Test", "Material"]]

,Contribution,Test,Material
0,pmdco2_tto_example_parallel,pmdao-tto-tt-S355-1_process,S355
1,pmdco2_tto_example_parallel,pmdao-tto-tt-S355-2_process,S355
2,pmdco2_tto_example_parallel,pmdao-tto-tt-S355-3_process,S355
3,pmdco2_tto_example_parallel,pmdao-tto-tt-S355-4_process,S355
4,pmdco2_tto_example_perpendicular,pmdao-tto-tt-S355-5_process,S355
5,pmdco2_tto_example_perpendicular,pmdao-tto-tt-S355-6_process,S355
6,pmdco2_tto_example_perpendicular,pmdao-tto-tt-S355-7_process,S355
7,pmdco2_tto_example_perpendicular,pmdao-tto-tt-S355-8_process,S355
8,pmdco2_tto_example_diagonal,pmdao-tto-tt-S355-10_process,S355
9,pmdco2_tto_example_diagonal,pmdao-tto-tt-S355-9_process,S355


**Interpretation:** the example has four parallel, four perpendicular and two diagonal tests. Reusing the query makes these separately managed contributions comparable.

## 3. What belongs together?

### Retrieve the selected research graphs

Download exactly the three contributions chosen in the settings. The returned file list becomes the assembly input, so unrelated endpoints and old files in `downloaded/` are excluded.

**Execution:** network reads and local file writes.

In [21]:
downloaded_dir.mkdir(parents=True, exist_ok=True)
source_files = []
for endpoint in selected_endpoints:
    source_path = downloaded_dir / f"{endpoint.dataset}.ttl"
    ontodocker.datasets.download_turtle(endpoint.dataset, filename=source_path)
    source_files.append(source_path)

### Combine the graphs into a publication artifact

Parse the selected Turtle files into one RDF graph and serialize the result. Shared identifiers connect compatible descriptions. This operation does not resolve conflicting meanings or add provenance for each statement.

In [22]:
from rdflib import Graph

publication_graph = Graph()
for source_path in source_files:
    publication_graph.parse(source_path, format="turtle")

publication_graph.serialize(destination=ttl_path, format="turtle")
pd.DataFrame([{
    "Source graphs": len(source_files),
    "Combined triples": len(publication_graph),
    "Artifact": ttl_path.name,
}])

,Source graphs,Combined triples,Artifact
0,3,3352,pmdco2_tto_recombined.ttl


**Interpretation:** the saved example combines three contributions into a 3,352-triple scientific description. The CSV measurements remain external resources referenced by the graph.

## 4. Which metadata can we reuse?

The combined artifact is both a publication resource and a source of descriptive metadata. We distinguish extracted facts, researcher choices and audit information.

`PublicationMetadata` carries the general scientific description. `DataportalMetadata` adds the portal's organization, visibility and resource settings. Review both before creating a record.

#### Access the Dataportal via `praeco`

In [23]:
from praeco import DataportalClient

dataportal = DataportalClient(api_token=dataportal_token)
dataportal.base_url

'https://dataportal.material-digital.de'

### Extract facts and inspect attribution

Dublin Core (`dc`) provides creator references, CSVW (`csvw`) links CSV resources, RDFS provides labels, and PMDco describes the tests.

Creator names must remain associated with the identifiers that actually describe them. We inspect the RDF mapping before constructing `Person` entries. If the mapping is ambiguous, names are retained without assigned ORCIDs and the original identifiers remain in audit metadata for review.

In [24]:
from datetime import date

import rdflib
from praeco import Person, PublicationMetadata

publication_graph = Graph().parse(ttl_path, format="turtle")
DC = rdflib.Namespace("http://purl.org/dc/elements/1.1/")
CSVW = rdflib.Namespace("http://www.w3.org/ns/csvw#")
PMD = rdflib.Namespace("https://w3id.org/pmd/co/")
RDFS = rdflib.Namespace("http://www.w3.org/2000/01/rdf-schema#")

In [25]:
creator_ids = sorted({str(obj) for obj in publication_graph.objects(None, DC.creator)})
creator_labels_by_id = {
    identifier: sorted({
        str(label) for label in publication_graph.objects(rdflib.URIRef(identifier), RDFS.label)
    })
    for identifier in creator_ids
}
creator_orcids = [
    identifier for identifier in creator_ids
    if urlsplit(identifier).hostname == "orcid.org"
]
creator_labels = sorted({label for labels in creator_labels_by_id.values() for label in labels})
creator_mapping_unambiguous = (
    bool(creator_ids)
    and all(len(labels) == 1 for labels in creator_labels_by_id.values())
    and len(creator_labels) == len(creator_ids)
)

csv_urls = sorted({str(obj) for obj in publication_graph.objects(None, CSVW.url)})
test_count = len(set(publication_graph.subjects(rdflib.RDF.type, PMD.TensileTest)))
orientation_query = S355queries.orientation()
orientation_index = orientation_query.query_vars.index("cut_orientation")
orientations = sorted({
    str(row[orientation_index]) for row in publication_graph.query(orientation_query.query)
})

display(pd.DataFrame([{
    "Triples": len(publication_graph), "Tensile tests": test_count,
    "CSV resources": len(csv_urls), "Orientations": "; ".join(orientations),
}]))
pd.DataFrame([
    {"Creator identifier": identifier, "Labels in the RDF": "; ".join(labels) or "No label"}
    for identifier, labels in creator_labels_by_id.items()
])

,Triples,Tensile tests,CSV resources,Orientations
0,3352,10,10,diagonal to rolling direction; in rolling direction; perpendicular...


,Creator identifier,Labels in the RDF
0,https://orcid.org/0000-0001-7192-7143,No label
1,https://orcid.org/0000-0002-3717-7104,Bernd Bayerlein; Jörg Waitelonis; Markus Schilling; Philipp von Ha...
2,https://orcid.org/0000-0002-7094-5371,No label
3,https://orcid.org/0000-0003-4971-3645,No label


In [26]:
if creator_mapping_unambiguous:
    creators = tuple(
        Person(
            name=labels[0],
            orcid=identifier.rstrip("/").rsplit("/", 1)[-1] if identifier in creator_orcids else None,
        )
        for identifier, labels in creator_labels_by_id.items()
    )
else:
    # The current graph attaches four names to one ORCID. Do not guess pairings.
    creators = tuple(Person(name=name) for name in creator_labels)
    print("Attribution needs review: names retained; ambiguous ORCIDs remain in audit metadata.")
if not creators:
    raise ValueError("No named creators were found. Supply reviewed creator metadata before continuing.")

metadata = PublicationMetadata(
    title="Recombined S355 tensile test RDF dataset",
    description=(
        "RDF/Turtle artifact assembled from three selected Ontodocker datasets "
        "for the PMD public infrastructure demonstrator. "
        f"The graph contains {test_count} tensile tests and references {len(csv_urls)} CSV resources. "
        f"Specimen orientations: {', '.join(orientations)}."
    ),
    creators=creators,
    publication_date=date.today(),
    keywords=("Platform MaterialDigital", "PMD", "S355", "tensile test", "RDF", "Turtle", "CSVW", "Ontodocker"),
    license="CC-BY-4.0",  # Confirm the intended reuse terms before publication.
)

display(pd.DataFrame([{
    "Title": metadata.title, "Publication date": str(metadata.publication_date),
    "License": metadata.license,
}]))
pd.DataFrame([{"Creator": person.name, "ORCID": person.orcid or "Unassigned"} for person in creators])

Attribution needs review: names retained; ambiguous ORCIDs remain in audit metadata.


,Title,Publication date,License
0,Recombined S355 tensile test RDF dataset,2026-09-24,CC-BY-4.0


,Creator,ORCID
0,Bernd Bayerlein,Unassigned
1,Jörg Waitelonis,Unassigned
2,Markus Schilling,Unassigned
3,Philipp von Hartrott,Unassigned


### Confirm the owning organization

The token determines where we may create a dataset. Select the intended organization by name in the settings, then check it against the available organizations. Taking the first result would hide a publication decision.

In [27]:
organizations = dataportal.action.call(
    "organization_list_for_user", {"permission": "create_dataset"}
)
organization_options = [
    {"name": organization.get("name"), "title": organization.get("title")}
    for organization in organizations
]
pd.DataFrame(organization_options, columns=["name", "title"])

,name,title
0,mpi-susmat,MPI-SusMat
1,pmd,PMD


In [28]:
eligible_organizations = {organization["name"] for organization in organization_options}
if publication_mode == "create" and owner_org not in eligible_organizations:
    raise RuntimeError(f"The token cannot create datasets in the selected organization: {owner_org}")

pd.DataFrame([{
    "Selected organization": owner_org,
    "Can create here": owner_org in eligible_organizations,
    "Publication mode": publication_mode,
}])

,Selected organization,Can create here,Publication mode
0,mpi-susmat,True,reuse


### Record audit information

A SHA-256 checksum identifies the artifact's exact bytes. Source files, graph size and type counts document the assembly result. These are useful audit details; complete provenance and long-term preservation need further decisions.

The compact preview below shows the publication choices. The full payload remains available as `publication_payload` for technical inspection.

In [29]:
import hashlib
import json
from collections import Counter

from praeco.services.dataportal.metadata import DataportalMetadata

ttl_bytes = ttl_path.read_bytes()
sha256 = hashlib.sha256(ttl_bytes).hexdigest()

namespaces = {
    prefix: str(namespace)
    for prefix, namespace in publication_graph.namespaces()
    if prefix
}

type_counts = Counter(
    str(obj) for obj in publication_graph.objects(None, rdflib.RDF.type)
)

audit_extras = {
    "source_file": str(ttl_path.relative_to(demo_dir)),
    "source_files": json.dumps(
        [str(path.relative_to(demo_dir)) for path in source_files],
        sort_keys=True,
        separators=(",", ":"),
    ),
    "sha256": sha256,
    "file_size_bytes": str(ttl_path.stat().st_size),
    "rdf_triple_count": str(len(publication_graph)),
    "rdf_namespaces": json.dumps(
        namespaces,
        sort_keys=True,
        separators=(",", ":"),
    ),
    "rdf_type_counts": json.dumps(
        dict(sorted(type_counts.items())),
        sort_keys=True,
        separators=(",", ":"),
    ),
    "creator_orcids": json.dumps(
        creator_orcids,
        sort_keys=True,
        separators=(",", ":"),
    ),
    "csv_resource_urls": json.dumps(
        csv_urls,
        sort_keys=True,
        separators=(",", ":"),
    ),
    "pmd_profile": "pmdco2-tto-recombined",
}

dataportal_metadata = DataportalMetadata(
    metadata=metadata,
    name=publication_name,
    owner_org=owner_org,
    private=False,
    extras=audit_extras,
)

publication_payload = dataportal_metadata.to_payload()
pd.DataFrame([
    {"Field": "Publication name", "Value": publication_name},
    {"Field": "Owning organization", "Value": owner_org},
    {"Field": "Visibility", "Value": "Public"},
    {"Field": "License", "Value": metadata.license},
    {"Field": "Sources", "Value": ", ".join(path.name for path in source_files)},
    {"Field": "Artifact bytes", "Value": str(len(ttl_bytes))},
    {"Field": "RDF triples", "Value": str(len(publication_graph))},
    {"Field": "SHA-256", "Value": sha256},
])

,Field,Value
0,Publication name,pmdco2-tto-recombined
1,Owning organization,mpi-susmat
2,Visibility,Public
3,License,CC-BY-4.0
4,Sources,"pmdco2_tto_example_parallel.ttl, pmdco2_tto_example_perpendicular...."
5,Artifact bytes,154685
6,RDF triples,3352
7,SHA-256,d0167d62ac5f51036cc4ad0f5dd7da0d3a25d0ed54e7789042959f02bdffc0c0


**Review pause:** the graph supplies counts, orientations and resource links. We choose the title, license, owning organization and visibility, and resolve attribution questions. In `reuse` mode this is a draft preview; it does not update the existing publication.

## 5. What can others access?

### Create a publication or inspect the existing one

**Execution:** network read in `reuse` mode. In `create` mode this cell creates a record, uploads the scientific Turtle file, and registers the external CSV URLs. It requires `publication_reviewed = True` in the settings after reviewing the draft.

A repeated creation with the same name stops before uploading again. If a publication only partly completes, inspect its resources before deciding how to recover; no automatic deletion, replacement or repair is performed here.

In [30]:
from praeco.services.ckan.exceptions import CkanApiError

if publication_mode == "reuse":
    published_dataset = dataportal.datasets.show(publication_name)
else:
    if not publication_reviewed:
        raise RuntimeError("Review the publication preview and attribution, then set publication_reviewed = True.")
    try:
        dataportal.datasets.show(publication_name)
    except CkanApiError as exc:
        error = (exc.payload or {}).get("error", {}) if isinstance(exc.payload, dict) else {}
        if not isinstance(error, dict) or error.get("__type") != "Not Found Error":
            raise
    else:
        raise RuntimeError("Publication already exists. Select reuse or a new name; no resources were added.")

    published_dataset = dataportal.datasets.create(dataportal_metadata)
    ttl_asset = dataportal.assets.upload_rdf(
        published_dataset, ttl_path, name=ttl_path.name,
        description="Combined RDF/Turtle scientific description.",
    )
    csv_assets = [
        dataportal.assets.create_url(
            published_dataset, url=url, name=local_name(url),
            description="CSV resource referenced by the scientific graph.", format="csv",
        )
        for url in csv_urls
    ]
    published_dataset = dataportal.datasets.show(publication_name)

pd.DataFrame([{
    "Publication": published_dataset.title,
    "Organization": published_dataset.raw.get("organization", {}).get("name", published_dataset.owner_org),
    "Public": not published_dataset.private,
    "Resources currently listed": len(published_dataset.raw.get("resources", [])),
    "Mode": publication_mode,
}])

,Publication,Organization,Public,Resources currently listed,Mode
0,Recombined S355 tensile test RDF dataset,mpi-susmat,True,32,reuse


### Inspect the publication's resources

The record connects four different outputs:

- **Catalog metadata:** a description of the publication, available as DCAT RDF.
- **Scientific artifact:** our uploaded Turtle file describing the tensile tests.
- **Measurement files:** links to the externally hosted CSV files.
- **Generated metadata:** portal-generated CSVW resources and a listed SPARQL endpoint.

On 24 September 2026, the endpoint exposed CSVW-related metadata, while the uploaded scientific artifact remained separately downloadable. An endpoint's presence does not establish which graph it contains; inspect its content explicitly.

In [31]:
published_dataset = dataportal.datasets.show(publication_name)
resources = published_dataset.raw.get("resources", [])
uploaded_ttl_matches = [resource for resource in resources if resource.get("name") == ttl_path.name]
if len(uploaded_ttl_matches) != 1:
    raise RuntimeError("Expected exactly one scientific Turtle artifact. Inspect the publication resources.")
uploaded_ttl_resource = uploaded_ttl_matches[0]
sparql_resources = [
    resource for resource in resources
    if str(resource.get("format", "")).strip().lower() == "sparql" and resource.get("url")
]
generated_turtle_resources = [
    resource for resource in resources
    if resource.get("wasGeneratedBy") and (
        "turtle" in str(resource.get("format", "")).lower()
        or "turtle" in str(resource.get("mimetype", "")).lower()
    )
]
dcat_metadata_urls = {
    format_name: dataportal.rdf.dataset_url(published_dataset, format=format_name)
    for format_name in ("ttl", "jsonld")
}

resource_links([
    ("Publication page", f"{dataportal.base_url}/dataset/{published_dataset.name}"),
    ("Uploaded scientific Turtle artifact", uploaded_ttl_resource["url"]),
    ("Catalog metadata (DCAT Turtle)", dcat_metadata_urls["ttl"]),
    ("Catalog metadata (DCAT JSON-LD)", dcat_metadata_urls["jsonld"]),
    *((f"SPARQL resource {index}", resource["url"]) for index, resource in enumerate(sparql_resources, 1)),
])
display(pd.DataFrame([
    {"Resource": resource.get("name"), "Format": resource.get("format"),
     "Portal generated": bool(resource.get("wasGeneratedBy"))}
    for resource in resources
]))

Resource,Link
Publication page,Open resource
Uploaded scientific Turtle artifact,Open resource
Catalog metadata (DCAT Turtle),Open resource
Catalog metadata (DCAT JSON-LD),Open resource
SPARQL resource 1,Open resource


,Resource,Format,Portal generated
0,pmdco2_tto_recombined.ttl,ttl,False
1,Zd2.csv,CSV,False
2,Zd3.csv,CSV,False
3,Zx1.csv,CSV,False
4,Zx2.csv,CSV,False
5,Zx3.csv,CSV,False
6,Zx4.csv,CSV,False
7,Zy1.csv,CSV,False
8,Zd2-metadata.jsonld,json-ld,True
9,Zy2.csv,CSV,False


### Verify the uploaded scientific artifact

Retrieve the uploaded file through the authenticated Dataportal client and compare its checksum with the **published record's** audit metadata. Then parse it locally and count tensile-test instances. This check verifies the uploaded artifact independently of the SPARQL endpoint.

A fresh RDF serialization may have different bytes while describing an equivalent graph. In `reuse` mode, the reference checksum belongs to the existing publication, not the newly assembled local draft.

**Execution:** network read and local computation only.

In [32]:
artifact_response = dataportal.request("GET", uploaded_ttl_resource["url"])
artifact_response.raise_for_status()
published_artifact_bytes = artifact_response.content
published_extras = {item["key"]: item["value"] for item in published_dataset.raw.get("extras", [])}
recorded_sha256 = published_extras.get("sha256")
if not recorded_sha256:
    raise RuntimeError("The publication has no recorded SHA-256 to compare with the download.")
downloaded_sha256 = hashlib.sha256(published_artifact_bytes).hexdigest()
checksum_matches = downloaded_sha256 == recorded_sha256
if not checksum_matches:
    raise RuntimeError("Downloaded artifact checksum differs from the publication record. Inspect before continuing.")

verified_graph = Graph().parse(data=published_artifact_bytes, format="turtle")
verified_test_count = len(set(verified_graph.subjects(rdflib.RDF.type, PMD.TensileTest)))
pd.DataFrame([{
    "Artifact": uploaded_ttl_resource["name"],
    "Checksum matches publication record": checksum_matches,
    "Scientific triples": len(verified_graph),
    "Tensile tests": verified_test_count,
}])

,Artifact,Checksum matches publication record,Scientific triples,Tensile tests
0,pmdco2_tto_recombined.ttl,True,3352,10


### Inspect endpoint content separately from the artifact

Query the listed endpoint's default graph for its triple count and RDF types, and enumerate its named graphs. Retrieve DCAT independently through the catalog RDF helper.

The earlier snapshot had 408 default-graph triples at the endpoint, compared with 3,352 triples in the scientific artifact and 321 in the DCAT response. They describe different content; their counts are not expected to agree. Portal-generated resources may appear asynchronously, so refresh the resource-list cell if an endpoint is not yet listed.

In [33]:
if sparql_resources:
    portal_sparql_endpoint = dataportal.sparql.endpoint(published_dataset)
    portal_graph_count = dataportal.sparql.query_df(
        portal_sparql_endpoint,
        "SELECT (COUNT(*) AS ?triples) WHERE { ?s ?p ?o }", ["triples"],
    )
    portal_types = dataportal.sparql.query_df(
        portal_sparql_endpoint,
        "SELECT ?type (COUNT(*) AS ?count) WHERE { ?s a ?type } GROUP BY ?type ORDER BY DESC(?count)",
        ["type", "count"],
    )
    portal_named_graphs = dataportal.sparql.query_df(
        portal_sparql_endpoint,
        "SELECT ?graph (COUNT(*) AS ?triples) WHERE { GRAPH ?graph { ?s ?p ?o } } GROUP BY ?graph",
        ["graph", "triples"],
    )
    display(portal_graph_count.rename(columns={"triples": "Endpoint default-graph triples"}))
    display(portal_types.assign(type=portal_types["type"].map(local_name)))
    display(portal_named_graphs.assign(graph=portal_named_graphs["graph"].map(local_name)))
else:
    print("No SPARQL resource is listed yet. Refresh the preceding resource-list cell later.")

dcat_metadata_graph = Graph().parse(
    data=dataportal.rdf.dataset(published_dataset, format="ttl"), format="turtle"
)
pd.DataFrame([{
    "Catalog record": published_dataset.title,
    "DCAT triples": len(dcat_metadata_graph),
    "Generated Turtle resources listed": len(generated_turtle_resources),
}])

,Endpoint default-graph triples
0,408


,type,count
0,Column,35
1,TableGroup,7
2,Unit,3
3,DerivedUnit,1
4,Activity,1
5,SoftwareAgent,1


,graph,triples
0,zd2-metadata.jsonld,66
1,zx1-metadata.jsonld,66
2,zx4-metadata.jsonld,66
3,zy1-metadata.jsonld,66
4,zd3-metadata.jsonld,66
5,zx2-metadata.jsonld,66
6,zx3-metadata.jsonld,66


,Catalog record,DCAT triples,Generated Turtle resources listed
0,Recombined S355 tensile test RDF dataset,321,10


## Discussion and presenter maintenance

The workflow connects contributions to their scientific context, reuses descriptive facts for publication, and exposes several forms of the result. Metadata review, attribution, licensing, versioning and preservation remain explicit responsibilities.

**Discussion:** which step would reduce repeated work in your research group? What would you need to add for long-term reuse?

### Cleanup outside the walkthrough

The following lines are deliberately commented out. Delete only demonstration resources that you intentionally created and no longer need. Reused collaborator datasets and existing publications should remain available. Review each explicit resource name before enabling a deletion.

In [34]:
# ontodocker.datasets.delete(ontodocker_dataset_name)  # Review ownership and the exact name first.

In [35]:
# dataportal.datasets.delete(publication_name)  # Review ownership and the exact name first.